In [1]:
import os

In [2]:
%pwd

'd:\\05 GIT\\17_TextSummarizer_NLP\\notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\05 GIT\\17_TextSummarizer_NLP'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config

In [16]:
import os
import urllib.request as request
from zipfile import ZipFile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size
import shutil

In [19]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
    
    def get_file(self):
        if not os.path.exists(self.config.local_data_file):
            shutil.copy("notebooks/data/data.zip", self.config.local_data_file)
            logger.info(f"File copied to: {self.config.local_data_file}")
        else:
            logger.info(f"File already exists: {self.config.local_data_file}")
    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL, filename=self.config.local_data_file
            )
            logger.info(f"File downloaded successfully: {filename}")
            logger.info(f"File size: {get_size(Path(filename))}")
        else:
            logger.info(f"File already exists: {self.config.local_data_file}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"File extracted successfully to: {unzip_path}")

In [21]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)

    #if local_data_file doesn't exist in artifacts directory
    if not os.path.exists(data_ingestion.config.local_data_file):
        #if file exists at notebooks/data/data.zip, copy the file
        if os.path.exists("notebooks/data/data.zip"):
            data_ingestion.get_file()
        #else download the file
        else:
            data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-06-07 00:56:22,147: INFO: common]: Yaml file: config\config.yaml loaded successfully
[2026-06-07 00:56:22,149: INFO: common]: Yaml file: params.yaml loaded successfully
[2026-06-07 00:56:22,149: INFO: common]: Directory created: artifacts
[2026-06-07 00:56:22,149: INFO: common]: Directory created: artifacts/data_ingestion
[2026-06-07 00:56:22,156: INFO: 1999065244]: File copied to: artifacts/data_ingestion/data.zip
[2026-06-07 00:56:22,263: INFO: 1999065244]: File extracted successfully to: artifacts/data_ingestion
